# Polymarket Observer — Exploratory Data Analysis

Explores collected data to understand distributions, data quality, and the
relationship between price features and market resolution.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit as sigmoid

from analysis.data_loader import (
    load_all_intervals, load_all_snapshots,
    join_snapshots_intervals, add_formula_features,
    TIMEFRAME_SETTINGS,
)

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
})

TF = '5m'  # Change to '15m' to analyze 15-minute markets

## 1. Load Data

In [ ]:
intervals = load_all_intervals(TF)
snapshots = load_all_snapshots(TF)

print(f'Intervals: {len(intervals)} rows, {sorted(intervals["asset"].unique())}')
print(f'Snapshots: {len(snapshots)} rows, {sorted(snapshots["asset"].unique())}')
print(f'\nResolution distribution:')
print(intervals['resolution'].value_counts())
print(f'\nPer-asset resolution:')
print(intervals.groupby('asset')['resolution'].value_counts().unstack(fill_value=0))

## 2. Data Quality

In [ ]:
# Chainlink data quality
print('=== Chainlink tick quality per asset ===')
for asset in sorted(intervals['asset'].unique()):
    sub = intervals[intervals['asset'] == asset]
    print(f"\n{asset.upper()}:")
    print(f"  Avg tick count: {sub['chainlink_tick_count'].mean():.0f}")
    print(f"  Avg gap count:  {sub['chainlink_gap_count'].mean():.1f}")
    print(f"  Max gap (ms):   {sub['chainlink_max_gap_ms'].max():.0f}")
    print(f"  Vol non-null:   {sub['realized_vol_20'].notna().sum()}/{len(sub)}")

print('\n=== Snapshot source breakdown ===')
print(snapshots['chainlink_source'].value_counts())
print()
print(snapshots['book_source'].value_counts())

In [ ]:
# Chainlink tick age distribution (live rows only)
live = snapshots[snapshots['chainlink_source'] == 'live']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(live['chainlink_tick_age_ms'].clip(upper=5000), bins=50, alpha=0.7)
axes[0].set_xlabel('Chainlink tick age (ms)')
axes[0].set_ylabel('Count')
axes[0].set_title('Chainlink tick freshness (clipped at 5s)')
axes[0].axvline(2000, color='red', linestyle='--', label='2s threshold')
axes[0].legend()

axes[1].hist(live['binance_tick_age_ms'].clip(upper=2000), bins=50, alpha=0.7, color='orange')
axes[1].set_xlabel('Binance tick age (ms)')
axes[1].set_ylabel('Count')
axes[1].set_title('Binance tick freshness (clipped at 2s)')

plt.tight_layout()
plt.show()

## 3. Delta Distributions

In [ ]:
resolved = intervals[intervals['resolution'].isin(['up', 'down'])].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Delta by resolution
for res, color in [('up', 'green'), ('down', 'red')]:
    sub = resolved[resolved['resolution'] == res]
    axes[0].hist(sub['delta'] * 100, bins=30, alpha=0.6, label=f'{res} (n={len(sub)})', color=color)
axes[0].set_xlabel('Delta (%)')
axes[0].set_ylabel('Count')
axes[0].set_title('Interval delta by resolution')
axes[0].axvline(0, color='black', linestyle='-', linewidth=0.5)
axes[0].legend()

# |delta| / vol ratio (the formula's core signal)
has_vol = resolved[resolved['realized_vol_20'].notna() & (resolved['realized_vol_20'] > 0)].copy()
has_vol['delta_vol_ratio'] = has_vol['abs_delta'] / has_vol['realized_vol_20']

for res, color in [('up', 'green'), ('down', 'red')]:
    sub = has_vol[has_vol['resolution'] == res]
    axes[1].hist(sub['delta_vol_ratio'], bins=30, alpha=0.6, label=f'{res} (n={len(sub)})', color=color)
axes[1].set_xlabel('|delta| / vol')
axes[1].set_ylabel('Count')
axes[1].set_title('Normalized delta magnitude by resolution')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nDelta stats (%):')
print(resolved.groupby('resolution')['delta'].describe().round(6) * 100)

## 4. Realized Volatility

In [ ]:
has_vol_all = intervals[intervals['realized_vol_20'].notna()].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Vol distribution per asset
for asset in sorted(has_vol_all['asset'].unique()):
    sub = has_vol_all[has_vol_all['asset'] == asset]
    axes[0].hist(sub['realized_vol_20'] * 100, bins=20, alpha=0.5, label=asset.upper())
axes[0].set_xlabel('Realized vol (%, 20-interval)')
axes[0].set_ylabel('Count')
axes[0].set_title('Realized volatility distribution')
axes[0].legend()

# |delta| vs vol scatter
resolved_vol = has_vol_all[has_vol_all['resolution'].isin(['up', 'down'])]
colors = resolved_vol['resolution'].map({'up': 'green', 'down': 'red'})
axes[1].scatter(resolved_vol['realized_vol_20'] * 100, resolved_vol['abs_delta'] * 100,
                c=colors, alpha=0.4, s=20)
axes[1].set_xlabel('Realized vol (%)')
axes[1].set_ylabel('|delta| (%)')
axes[1].set_title('|delta| vs realized vol (green=up, red=down)')
# Add diagonal for delta/vol = 1
lim = max(axes[1].get_xlim()[1], axes[1].get_ylim()[1])
axes[1].plot([0, lim], [0, lim], 'k--', alpha=0.3, label='|delta|/vol = 1')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Token Pricing Within Intervals

In [ ]:
# Pick a few example intervals to visualize token price evolution
btc_ints = intervals[(intervals['asset'] == 'btc') & (intervals['resolution'].isin(['up', 'down']))]

# Pick 3 up and 3 down (or fewer if not enough)
up_ids = btc_ints[btc_ints['resolution'] == 'up']['interval_id'].iloc[:3].tolist()
down_ids = btc_ints[btc_ints['resolution'] == 'down']['interval_id'].iloc[:3].tolist()
example_ids = up_ids + down_ids

btc_snaps = snapshots[(snapshots['asset'] == 'btc') & (snapshots['interval_id'].isin(example_ids))]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, iid in enumerate(example_ids):
    ax = axes[idx // 3][idx % 3]
    sub = btc_snaps[btc_snaps['interval_id'] == iid].sort_values('seconds_into_interval')
    res = btc_ints[btc_ints['interval_id'] == iid]['resolution'].iloc[0]

    ax.plot(sub['seconds_into_interval'], sub['up_token_ask'], label='Up ask', color='green', alpha=0.7)
    ax.plot(sub['seconds_into_interval'], sub['down_token_ask'], label='Down ask', color='red', alpha=0.7)
    ax.set_xlabel('Seconds')
    ax.set_ylabel('Token price')
    ax.set_title(f'{iid.split("-")[-1]} [{res.upper()}]')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
    # Shade trading window
    window_start = TIMEFRAME_SETTINGS[TF]['duration_s'] - TIMEFRAME_SETTINGS[TF]['trading_window_s']
    ax.axvspan(window_start, TIMEFRAME_SETTINGS[TF]['duration_s'], alpha=0.1, color='blue')

plt.suptitle('Token price evolution (blue shading = trading window)', y=1.02)
plt.tight_layout()
plt.show()

## 6. Fee Kill Zone

In [ ]:
# Token price distribution during trading window
settings = TIMEFRAME_SETTINGS[TF]
window_start = settings['duration_s'] - settings['trading_window_s']

tw = snapshots[
    (snapshots['seconds_into_interval'] >= window_start) &
    (snapshots['chainlink_source'] == 'live') &
    (snapshots['book_source'] == 'live')
].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Up token ask distribution
axes[0].hist(tw['up_token_ask'].dropna(), bins=50, alpha=0.7, color='green')
axes[0].set_xlabel('Up token ask price')
axes[0].set_ylabel('Count')
axes[0].set_title('Up token ask during trading window')
axes[0].axvline(0.65, color='orange', linestyle='--', label='Gate: $0.65')
axes[0].axvline(0.85, color='orange', linestyle='--', label='Gate: $0.85')
axes[0].legend()

# Fee curve overlay
prices = np.linspace(0.01, 0.99, 100)
fees = 0.0624 * prices * (1 - prices)
fee_pct = fees / prices * 100  # fee as % of trade

axes[1].plot(prices, fees, 'b-', label='Fee per share')
ax2 = axes[1].twinx()
ax2.plot(prices, fee_pct, 'r--', label='Fee % of trade value')
axes[1].set_xlabel('Token price')
axes[1].set_ylabel('Fee per share ($)', color='blue')
ax2.set_ylabel('Fee % of trade value', color='red')
axes[1].set_title('Taker fee structure')
axes[1].axvspan(0.65, 0.85, alpha=0.1, color='green', label='Target zone')
axes[1].legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

print(f'Trading window rows: {len(tw)}')
print(f'Up ask in $0.65-$0.85: {((tw["up_token_ask"] >= 0.65) & (tw["up_token_ask"] <= 0.85)).sum()} ({((tw["up_token_ask"] >= 0.65) & (tw["up_token_ask"] <= 0.85)).mean()*100:.1f}%)')

## 7. Book Depth & Spread

In [ ]:
live_book = snapshots[snapshots['book_source'] == 'live'].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Depth at level 1 (best ask)
axes[0].hist(live_book['up_depth_1'].clip(upper=500), bins=50, alpha=0.6, label='Up', color='green')
axes[0].hist(live_book['down_depth_1'].clip(upper=500), bins=50, alpha=0.6, label='Down', color='red')
axes[0].set_xlabel('Depth at best ask ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('Level 1 depth (clipped at $500)')
axes[0].legend()

# Spread
axes[1].hist(live_book['spread_up'].dropna().clip(upper=0.1), bins=50, alpha=0.6, label='Up', color='green')
axes[1].hist(live_book['spread_down'].dropna().clip(upper=0.1), bins=50, alpha=0.6, label='Down', color='red')
axes[1].set_xlabel('Spread (ask - bid)')
axes[1].set_ylabel('Count')
axes[1].set_title('Bid-ask spread')
axes[1].legend()

# Depth by seconds into interval
live_book['sec_bin'] = (live_book['seconds_into_interval'] // 30) * 30
depth_by_sec = live_book.groupby('sec_bin')['up_depth_1'].median()
axes[2].plot(depth_by_sec.index, depth_by_sec.values, color='green', label='Up depth 1 (median)')
depth_down = live_book.groupby('sec_bin')['down_depth_1'].median()
axes[2].plot(depth_down.index, depth_down.values, color='red', label='Down depth 1 (median)')
axes[2].set_xlabel('Seconds into interval')
axes[2].set_ylabel('Median depth at best ask')
axes[2].set_title('Depth evolution over interval')
axes[2].axvline(window_start, color='blue', linestyle='--', alpha=0.5, label='Trading window')
axes[2].legend()

plt.tight_layout()
plt.show()

## 8. Cross-Asset Correlation

In [ ]:
# Pivot intervals to get delta per asset per start_ts
resolved_only = intervals[intervals['resolution'].isin(['up', 'down'])].copy()
delta_pivot = resolved_only.pivot_table(index='start_ts', columns='asset', values='delta')

corr = delta_pivot.corr()
print('Delta correlation matrix:')
print(corr.round(3))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels([c.upper() for c in corr.columns])
ax.set_yticklabels([c.upper() for c in corr.columns])
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=12)
plt.colorbar(im)
ax.set_title(f'Cross-asset delta correlation ({TF})')
plt.tight_layout()
plt.show()

# Resolution agreement
res_pivot = resolved_only.pivot_table(index='start_ts', columns='asset', values='resolution', aggfunc='first')
all_same = (res_pivot.nunique(axis=1) == 1).mean()
print(f'\nAll 4 assets resolve same direction: {all_same:.1%} of intervals')

## 9. Formula Preview

Compute the sigmoid confidence for each snapshot row using starting parameters
and see how it relates to the actual outcome.

In [ ]:
# Prepare joined data
merged = join_snapshots_intervals(snapshots, intervals)
merged = add_formula_features(merged, TF)

# Filter to: live data, in trading window, has vol
formula_df = merged[
    (merged['chainlink_source'] == 'live') &
    (merged['in_trading_window']) &
    (merged['realized_vol_20'].notna()) &
    (merged['realized_vol_20'] > 0) &
    (merged['resolution'].isin(['up', 'down']))
].copy()

print(f'Rows for formula analysis: {len(formula_df)}')

# Starting parameters
a, b, offset = 5, 3, 4

# Compute confidence
formula_df['signal'] = a * formula_df['abs_delta'] / formula_df['realized_vol_20'] + b * formula_df['window_fraction'] - offset
formula_df['confidence'] = sigmoid(formula_df['signal'])

# Direction: positive delta -> up confidence, negative -> down confidence
formula_df['predicted_dir'] = np.where(formula_df['delta'] >= 0, 'up', 'down')
formula_df['correct'] = formula_df['predicted_dir'] == formula_df['resolution']

print(f'\nOverall accuracy (delta direction = resolution): {formula_df["correct"].mean():.1%}')
print(f'Mean confidence: {formula_df["confidence"].mean():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confidence distribution by correctness
for correct, label, color in [(True, 'Correct', 'green'), (False, 'Wrong', 'red')]:
    sub = formula_df[formula_df['correct'] == correct]
    axes[0].hist(sub['confidence'], bins=30, alpha=0.6, label=f'{label} (n={len(sub)})', color=color)
axes[0].set_xlabel('Confidence')
axes[0].set_ylabel('Count')
axes[0].set_title('Confidence distribution')
axes[0].legend()

# Confidence vs seconds into trading window
sample = formula_df.sample(min(5000, len(formula_df)), random_state=42)
colors = sample['correct'].map({True: 'green', False: 'red'})
axes[1].scatter(sample['window_elapsed'], sample['confidence'], c=colors, alpha=0.1, s=5)
axes[1].set_xlabel('Seconds into trading window')
axes[1].set_ylabel('Confidence')
axes[1].set_title('Confidence vs time (green=correct, red=wrong)')

# Calibration: binned confidence vs actual accuracy
formula_df['conf_bin'] = pd.cut(formula_df['confidence'], bins=10)
cal = formula_df.groupby('conf_bin', observed=True)['correct'].agg(['mean', 'count'])
cal = cal[cal['count'] >= 10]  # need minimum samples

bin_centers = [interval.mid for interval in cal.index]
axes[2].plot(bin_centers, cal['mean'], 'bo-', markersize=8)
axes[2].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect calibration')
axes[2].set_xlabel('Predicted confidence')
axes[2].set_ylabel('Actual accuracy')
axes[2].set_title('Calibration curve')
axes[2].set_xlim(0, 1)
axes[2].set_ylim(0, 1)
axes[2].legend()

# Annotate with counts
for x, row in zip(bin_centers, cal.itertuples()):
    axes[2].annotate(f'n={row.count}', (x, row.mean), textcoords='offset points',
                     xytext=(0, 10), ha='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Entry simulation: would the formula trigger entries?
formula_df['entry_threshold'] = formula_df['up_token_ask'] + formula_df['fee_up']
formula_df['would_enter_up'] = (
    (formula_df['confidence'] > formula_df['entry_threshold']) &
    (formula_df['delta'] >= 0) &
    (formula_df['up_token_ask'] >= 0.65) &
    (formula_df['up_token_ask'] <= 0.85)
)

formula_df['entry_threshold_down'] = formula_df['down_token_ask'] + formula_df['fee_down']
formula_df['would_enter_down'] = (
    (formula_df['confidence'] > formula_df['entry_threshold_down']) &
    (formula_df['delta'] < 0) &
    (formula_df['down_token_ask'] >= 0.65) &
    (formula_df['down_token_ask'] <= 0.85)
)

entries_up = formula_df[formula_df['would_enter_up']]
entries_down = formula_df[formula_df['would_enter_down']]

print(f'Would-enter-up rows: {len(entries_up)} across {entries_up["interval_id"].nunique()} intervals')
print(f'Would-enter-down rows: {len(entries_down)} across {entries_down["interval_id"].nunique()} intervals')

if len(entries_up) > 0:
    print(f'\nUp entries — resolution accuracy: {(entries_up["resolution"] == "up").mean():.1%}')
    print(f'  Avg confidence at entry: {entries_up["confidence"].mean():.3f}')
    print(f'  Avg token ask at entry: ${entries_up["up_token_ask"].mean():.3f}')

if len(entries_down) > 0:
    print(f'\nDown entries — resolution accuracy: {(entries_down["resolution"] == "down").mean():.1%}')
    print(f'  Avg confidence at entry: {entries_down["confidence"].mean():.3f}')
    print(f'  Avg token ask at entry: ${entries_down["down_token_ask"].mean():.3f}')

## 10. Basis Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Basis at open and close
resolved_only = intervals[intervals['resolution'].isin(['up', 'down'])].copy()

axes[0].hist(resolved_only['open_basis_bps'], bins=30, alpha=0.6, label='Open basis')
axes[0].hist(resolved_only['close_basis_bps'], bins=30, alpha=0.6, label='Close basis')
axes[0].set_xlabel('Basis (bps)')
axes[0].set_ylabel('Count')
axes[0].set_title('Chainlink-Binance basis at interval boundaries')
axes[0].legend()

# Basis vs delta — does high basis predict anything?
colors = resolved_only['resolution'].map({'up': 'green', 'down': 'red'})
axes[1].scatter(resolved_only['close_basis_bps'], resolved_only['delta'] * 100,
                c=colors, alpha=0.5, s=20)
axes[1].set_xlabel('Close basis (bps)')
axes[1].set_ylabel('Delta (%)')
axes[1].set_title('Basis vs delta (green=up, red=down)')

plt.tight_layout()
plt.show()

print(f'Open basis: mean={resolved_only["open_basis_bps"].mean():.2f} bps, median={resolved_only["open_basis_bps"].median():.2f} bps')
print(f'Close basis: mean={resolved_only["close_basis_bps"].mean():.2f} bps, median={resolved_only["close_basis_bps"].median():.2f} bps')